# NB08 — Musanga satellite detectability | V4 native-reference rebuild

**Scientific target:** upscale NB07 V4 UAV-derived Musanga probability / fractional cover to satellite predictors. This notebook is a **clean V4 replacement**, not a continuation of the V3_1 satellite analysis.

- **UAV target:** six V4 NB07 georeferenced two-band mosaics (band 1 probability, band 2 binary at p≥0.5). The mosaics are already clipped by the NB07 ROI100 + RGB/DSM/MS validity policy. Values outside it remain NoData.
- **Satellite inputs:** existing source *raster products* only: annual Sentinel-2, annual embeddings, and three-year quarterly Sentinel-2. These can reside in the legacy `Outputs/V3_1/04_satellite_predictors/input_layers` input directory; **no V3_1 samples, targets, models, metrics, maps, or conclusions are imported**.
- **Primary validation:** strict leave-one-**forest**-out (Baego, MOS, Yangambi, Yoko), pooled MOS flight groups as one held-out forest. Equal-forest training and equal-forest reporting.
- **Scale comparison:** 50, 100, 125 m by default; 30 m may be added as an exploratory sensitivity test.
- **Interpretation:** compare source-derived satellite predictor families and explicitly report MOS >40% cover saturation. Site-holdout transfer may remain poor even if spatially held-out within-site skill is useful.
- **No Earth Engine exports or whole-region inference** in this validation notebook; do not claim an operational Kisangani map from these tests.

Run **00 → 04** in CPU Colab. Products are versioned under `Final/Outputs/v4/08_satellite_detectability_new_uav`. Input data are never overwritten. Sampling is cached for resumability. The previous notebook outputs are not reused.


## 00 — Settings
The 2020 Yoko satellite window follows the *available three-year raster product*, which must be checked against the actual Yoko UAV acquisition year before biological interpretation. Do not silently substitute 2021 products. Site/year matching is audited in Section 01.


In [ ]:
# ====================== USER SETTINGS ======================
SCALES_M = [50,100,125]   # add 30 only after checking 50 m
MAX_CELLS_PER_MOSAIC = 260
MAX_TRIALS_PER_MOSAIC = 2600
MIN_UAV_VALID_FRACTION = 0.90
SATELLITE_LOCAL_STAGE = True
LOCAL_STAGE_MAX_FRACTION_OF_FREE_DISK = .65
SEED = 20261122
MODEL_ITERATIONS = 180
TRAIN_CAP_PER_FOREST = 500
HIGH_COVER = 0.40
FEATURE_SETS = {
    'EMBEDDING': ['EMB_ANNUAL'],
    'S2_ANNUAL': ['RAW_ANNUAL'],
    'S2_SEASONAL': ['S2_QUARTERLY_3YR'],
    'EMB_PLUS_ANNUAL': ['EMB_ANNUAL','RAW_ANNUAL'],
    'EMB_PLUS_SEASONAL': ['EMB_ANNUAL','S2_QUARTERLY_3YR'],
    'ALL': ['EMB_ANNUAL','RAW_ANNUAL','S2_QUARTERLY_3YR'],
}
RUN_MODELS = True
FORCE_SAMPLES = False
# ==========================================================


## 01 — Audit six V4 UAV mosaics and independent satellite raster inputs
This is a hard-stop audit: required input layers, years, CRS, band count, finite valid-value coverage and output metadata must be consistent. Source rasters are staged to local SSD **one forest at a time**, only if there is room. No fallback to V3_1 target tables.


In [ ]:
%pip -q install rasterio pandas pyarrow scikit-learn matplotlib scipy
from pathlib import Path
from contextlib import ExitStack
from rasterio.windows import from_bounds,Window
from rasterio.enums import Resampling
import rasterio,numpy as np,pandas as pd
import hashlib,json,shutil,os,math,time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
FINAL=next((p for p in [
    Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
    Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL is not None,'Musanga/Final folder unavailable'
V4=FINAL/'Outputs/v4'
OUT=V4/'08_satellite_detectability_new_uav'
OUT.mkdir(parents=True,exist_ok=True)
UAV=V4/'07_RGB_MS_100m_ROI_complete_sensors_v4/reconstructed_mosaics'
SAT=FINAL/'Outputs/V3_1/04_satellite_predictors/input_layers'
assert SAT.is_dir(),f'Satellite source raster collection absent: {SAT}'
FORESTS=['Baego','MOS','Yangambi','Yoko']
GROUPS={
 'Baego':['Baego'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef',
        'Mos_3_4_transparent_mosaic_group1_georef',
        'Mos_5_6_transparent_mosaic_group1_georef'],
 'Yangambi':['Yangambi_georef'],
 'Yoko':['Yoko_kvo_georef'],
}
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
WINDOWS={'Baego':(2021,2023),'MOS':(2022,2024),
         'Yangambi':(2022,2024),'Yoko':(2018,2020)}
FAMILIES=['EMB_ANNUAL','RAW_ANNUAL','S2_QUARTERLY_3YR']
uav_paths={group:UAV/f'{group}_Musanga_probability_binary_v4.tif'
           for groups in GROUPS.values() for group in groups}
def satellite_name(site,family):
 if family=='S2_QUARTERLY_3YR':
  a,b=WINDOWS[site]
  return f'{site}_S2_QUARTERLY_3YR_{a}_{b}_10m.tif'
 return f'{site}_{family}_{YEARS[site]}_10m.tif'
sat_paths={(site,fam):SAT/fam/satellite_name(site,fam)
           for site in FORESTS for fam in FAMILIES}
records=[]
for group,p in uav_paths.items():
 assert p.is_file(),f'Missing V4 UAV mosaic: {p}'
 with rasterio.open(p) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs is not None,(group,ds.count,ds.nodata)
  assert all(t=='float32' for t in ds.dtypes),(group,ds.dtypes)
  records.append(dict(type='UAV V4',site=group,bands=ds.count,
                      width=ds.width,height=ds.height,crs=str(ds.crs),size_MB=round(p.stat().st_size/1e6)))
for (site,fam),p in sat_paths.items():
 assert p.is_file(),f'Missing source satellite raster: {p}'
 with rasterio.open(p) as ds:
  assert ds.crs is not None and ds.count>0,(site,fam)
  records.append(dict(type=fam,site=site,bands=ds.count,
                      width=ds.width,height=ds.height,crs=str(ds.crs),size_MB=round(p.stat().st_size/1e6)))
display(pd.DataFrame(records))
print('Yoko satellite reference year:',YEARS['Yoko'],
      '— verify acquisition alignment before interpretation')
print('V4-only target directory:',OUT)


## 02 — Build georeferenced V4 samples at multiple physical supports

Sampling grids are anchored to projected map coordinates, not to mosaic-array indices. Randomly selected grid cells must pass the **native UAV NoData coverage** threshold and all three satellite-family validity checks. A missing valid pixel is never treated as absence. Per-scale samples are cached with an input-identity fingerprint. Sampling is intentionally capped per *mosaic* and forest-balanced when fitting. Report low prevalence and high-cover counts before interpreting LOSO metrics.

**Performance note:** UAV reads are local-window reads from the completed V4 mosaics, not a full mosaic load; satellite layers are copied once per forest to local SSD when feasible. Expected time depends on mounted Drive read caching.


In [ ]:
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.windows import bounds as window_bounds
LOCAL=Path('/content/musanga_nb08_satellite')
LOCAL.mkdir(parents=True,exist_ok=True)
def input_fingerprint():
 h=hashlib.sha256()
 for p in list(uav_paths.values())+list(sat_paths.values()):
  st=p.stat()
  h.update(f'{p}/{st.st_size}/{st.st_mtime_ns}\n'.encode())
 h.update(json.dumps(dict(scales=SCALES_M,max_cells=MAX_CELLS_PER_MOSAIC,
    trials=MAX_TRIALS_PER_MOSAIC,valid=MIN_UAV_VALID_FRACTION,seed=SEED),sort_keys=True).encode())
 return h.hexdigest()[:20]
KEY=input_fingerprint()
def staged_sources(site):
 # Staging only reusable original satellite rasters. Never stage V3 targets/models.
 original={f:sat_paths[(site,f)] for f in FAMILIES}
 if not SATELLITE_LOCAL_STAGE:return original
 size=sum(p.stat().st_size for p in original.values())
 free=shutil.disk_usage(LOCAL).free
 if size>free*LOCAL_STAGE_MAX_FRACTION_OF_FREE_DISK:
  print(f'[{site}] Insufficient local space ({size/1e9:.1f} GB); using Drive source rasters')
  return original
 site_dir=LOCAL/site
 site_dir.mkdir(parents=True,exist_ok=True)
 result={}
 for fam,p in original.items():
  dest=site_dir/p.name
  if not dest.exists() or dest.stat().st_size!=p.stat().st_size:
   if dest.exists():dest.unlink()
   print(f'[{site}] Stage {fam}: {p.stat().st_size/1e6:.0f} MB',flush=True)
   tmp=dest.with_name(dest.name+'.copying')
   tmp.unlink(missing_ok=True)
   shutil.copyfile(p,tmp)
   assert tmp.stat().st_size==p.stat().st_size
   os.replace(tmp,dest)
  result[fam]=dest
 return result
def raster_mean(ds,bounds):
 # Out-of-bounds satellite support and missing bands are rejected, not zero-filled.
 l,b,r,t=bounds
 if not (ds.bounds.left<=l and ds.bounds.right>=r and
         ds.bounds.bottom<=b and ds.bounds.top>=t):return None
 win=from_bounds(l,b,r,t,transform=ds.transform)
 win=win.round_offsets().round_lengths()
 if win.width<=0 or win.height<=0:return None
 arr=ds.read(window=win,masked=True)
 if np.ma.is_masked(arr):
  fraction=np.mean(~np.ma.getmaskarray(arr).reshape(arr.shape[0],-1),axis=1)
  if np.any(fraction<.85):return None
  data=arr.mean(axis=(1,2)).filled(np.nan)
 else:data=arr.mean(axis=(1,2))
 return np.asarray(data,dtype=np.float32) if np.isfinite(data).all() else None
def uav_target(ds,bounds):
 l,b,r,t=bounds
 if not (ds.bounds.left<=l and ds.bounds.right>=r and
         ds.bounds.bottom<=b and ds.bounds.top>=t):return None
 win=from_bounds(l,b,r,t,transform=ds.transform)
 # Keep precise coverage; cap memory per native-resolution sample.
 win=win.round_offsets().round_lengths()
 if win.width<=0 or win.height<=0 or win.width*win.height>16_000_000:return None
 arr=ds.read([1,2],window=win,masked=True)
 valid=~np.ma.getmaskarray(arr[0])
 if valid.mean()<MIN_UAV_VALID_FRACTION:return None
 a=np.asarray(arr[0])[valid];b=np.asarray(arr[1])[valid]
 if not (np.isfinite(a).all() and np.isfinite(b).all()):return None
 if not ((a>=0).all() and (a<=1).all() and np.isin(b,[0,1]).all()):return None
 return float(a.mean()),float(b.mean()),float(valid.mean())
def candidate_centers(bounds,scale,rng,n):
 # Spatial grid is globally anchored to 0 in projected metres.
 c0=math.ceil(bounds.left/scale);c1=math.floor(bounds.right/scale)-1
 r0=math.ceil(bounds.bottom/scale);r1=math.floor(bounds.top/scale)-1
 if c1<c0 or r1<r0:return []
 candidates=set()
 for j in range(n*3):
  if len(candidates)>=n:break
  candidates.add((int(rng.integers(c0,c1+1)),int(rng.integers(r0,r1+1))))
 return [((x+.5)*scale,(y+.5)*scale) for x,y in candidates]
def sample_group(group,site,scale,satellite):
 rng=np.random.default_rng(SEED+scale*13+sum(ord(c) for c in group))
 rows=[]
 with ExitStack() as stack:
  u=stack.enter_context(rasterio.open(uav_paths[group]))
  # Assert genuine metric support (not geographic degree scales).
  assert u.crs.is_projected and abs(u.transform.a)<scale,(group,u.crs,u.transform)
  readers={}
  for fam,path in satellite.items():
   raw=stack.enter_context(rasterio.open(path))
   if raw.crs!=u.crs:
    readers[fam]=stack.enter_context(WarpedVRT(raw,crs=u.crs,
                     resampling=Resampling.bilinear))
   else:readers[fam]=raw
  points=candidate_centers(u.bounds,scale,rng,MAX_TRIALS_PER_MOSAIC)
  for j,(x,y) in enumerate(points):
   if len(rows)>=MAX_CELLS_PER_MOSAIC:break
   bbox=(x-scale/2,y-scale/2,x+scale/2,y+scale/2)
   target=uav_target(u,bbox)
   if target is None:continue
   feat={fam:raster_mean(ds,bbox) for fam,ds in readers.items()}
   if any(v is None for v in feat.values()):continue
   rec=dict(site=site,group=group,scale_m=scale,x_m=x,y_m=y,
            mean_probability=target[0],cover_fraction_p05=target[1],
            uav_valid_fraction=target[2])
   for fam,v in feat.items():
    for k,value in enumerate(v):
     rec[f'{fam}__{k+1:03d}']=float(value)
   rows.append(rec)
   if len(rows)%100==0:print(f'[{site}/{group} {scale}m] {len(rows)} samples',flush=True)
 print(f'[{group} {scale}m] {len(rows)} valid V4 cells from {len(points)} candidates',flush=True)
 return rows
all_scales=[]
for scale in SCALES_M:
 dest=OUT/f'02_V4_uav_satellite_cells_{scale}m_{KEY}.parquet'
 if dest.exists() and not FORCE_SAMPLES:
  frame=pd.read_parquet(dest)
  print('CACHE HIT',dest.name,len(frame),flush=True)
 else:
  pieces=[]
  for site in FORESTS:
   satellite=staged_sources(site)
   for group in GROUPS[site]:
    pieces+=sample_group(group,site,scale,satellite)
  assert pieces,f'No valid V4 samples at {scale} m'
  frame=pd.DataFrame(pieces)
  assert frame.site.nunique()==4,('Missing site support',scale,frame.site.value_counts())
  frame.to_parquet(dest,index=False)
  print('SAVED',dest,flush=True)
 all_scales.append(frame)
samples=pd.concat(all_scales,ignore_index=True)
summary=samples.groupby(['scale_m','site']).agg(
 n=('mean_probability','size'),mean_prob=('mean_probability','mean'),
 cover_40plus=('mean_probability',lambda x:int((x>=HIGH_COVER).sum())),
 valid_uav=('uav_valid_fraction','mean')).reset_index()
display(summary.round(3))
summary.to_csv(OUT/f'02_V4_support_summary_{KEY}.csv',index=False)


## 03 — Strict leave-one-forest-out: feature ablations and resolution test

All four forests are treated as independent geographic domains. **MOS 1–2, 3–4 and 5–6 are always held out together.** Training is capped at equal numbers per forest. Report balanced R² and MAE as well as per-site errors. A poor MOS high-cover estimate is not rescued by pooled low-cover averages. No V3_1 trained model is imported.


In [ ]:
def score(y,p):
 y=np.asarray(y,float);p=np.asarray(p,float);mask=y>=HIGH_COVER
 return dict(n=len(y),R2=float(r2_score(y,p)) if len(y)>1 and np.std(y)>1e-8 else np.nan,
  MAE_pp=float(np.mean(abs(p-y))*100),
  RMSE_pp=float(np.sqrt(np.mean((p-y)**2))*100),
  bias_pp=float(np.mean(p-y)*100),
  high_n=int(mask.sum()),
  high_bias_pp=float(np.mean(p[mask]-y[mask])*100) if mask.any() else np.nan,
  high_MAE_pp=float(np.mean(abs(p[mask]-y[mask]))*100) if mask.any() else np.nan)
def capped_train(frame,fold_seed):
 selected=[]
 for site,chunk in frame.groupby('site',sort=True):
  selected.append(chunk.sample(n=min(TRAIN_CAP_PER_FOREST,len(chunk)),
                              random_state=fold_seed))
 return pd.concat(selected,ignore_index=True)
metrics=[];predictions=[]
if RUN_MODELS:
 for scale,frame in samples.groupby('scale_m',sort=True):
  for family,groups in FEATURE_SETS.items():
   feature_cols=[c for c in frame.columns if any(c.startswith(f+'__') for f in groups)]
   assert feature_cols,(scale,family)
   for forest in FORESTS:
    train=capped_train(frame[frame.site!=forest],SEED+scale)
    test=frame[frame.site==forest].copy()
    assert len(train)>30 and len(test)>20,(scale,forest,len(train),len(test))
    # Balanced by forest, rather than by MOS mosaic count.
    weights=1/train.site.map(train.site.value_counts()).to_numpy(float)
    weights/=weights.mean()
    model=HistGradientBoostingRegressor(
      max_iter=MODEL_ITERATIONS,learning_rate=.06,max_leaf_nodes=18,
      min_samples_leaf=30,l2_regularization=2,early_stopping=False,
      random_state=SEED+scale)
    model.fit(train[feature_cols].to_numpy(np.float32),
              train.mean_probability.to_numpy(np.float32),sample_weight=weights)
    p=np.clip(model.predict(test[feature_cols].to_numpy(np.float32)),0,1)
    y=test.mean_probability.to_numpy(float)
    metrics.append(dict(scale_m=scale,family=family,held_out=forest,**score(y,p)))
    test=test[['site','group','scale_m','x_m','y_m','mean_probability']].copy()
    test['predicted']=p;test['family']=family
    predictions.append(test)
    print(f'{scale:3d}m {family:19s} holdout={forest:8s} '
          f'MAE={metrics[-1]["MAE_pp"]:.1f} pp, R2={metrics[-1]["R2"]:.2f}',flush=True)
 results=pd.DataFrame(metrics)
 held=pd.concat(predictions,ignore_index=True)
 results.to_csv(OUT/f'03_V4_LOSO_by_forest_{KEY}.csv',index=False)
 held.to_parquet(OUT/f'03_V4_LOSO_predictions_{KEY}.parquet',index=False)
 macro=results.groupby(['scale_m','family']).agg(
   macro_MAE_pp=('MAE_pp','mean'),
   macro_RMSE_pp=('RMSE_pp','mean'),
   macro_bias_pp=('bias_pp','mean'),
   mean_forest_R2=('R2','mean'),
   high_bias_pp=('high_bias_pp','mean'),
   forests=('held_out','nunique')).reset_index()
 assert (macro.forests==4).all()
 macro.to_csv(OUT/f'03_V4_LOSO_equal_forest_{KEY}.csv',index=False)
 display(macro.sort_values(['scale_m','macro_MAE_pp']).round(3))
else:
 print('RUN_MODELS=False: only fresh V4 sampling was performed.')


## 04 — Publication-oriented held-out diagnostics
Equal-forest summary, per-forest performance and held-out scatterplots (each forest displayed separately; MOS not allowed to visually dominate the others). Include one-to-one line, 40% cover reference and explicit high-cover sample counts. These are **strict external-forest** predictions, not spatially held-out circles. Re-run only after Section 03.


In [ ]:
if RUN_MODELS:
 import matplotlib.pyplot as plt
 for scale in SCALES_M:
  fig,axes=plt.subplots(1,4,figsize=(14,3.5),sharex=True,sharey=True)
  for ax,site in zip(axes,FORESTS):
   part=held[(held.scale_m==scale)&(held.family=='ALL')&(held.site==site)]
   if part.empty:
    ax.set_visible(False);continue
   ax.scatter(part.mean_probability*100,part.predicted*100,s=8,alpha=.40,rasterized=True)
   ax.plot([0,100],[0,100],c='black',lw=1,ls='--')
   ax.axvline(HIGH_COVER*100,c='gray',lw=.8,ls=':')
   stat=score(part.mean_probability,part.predicted)
   ax.set_title(f'{site} (n={len(part)})\nR²={stat["R2"]:.2f}; MAE={stat["MAE_pp"]:.1f} pp\n'
                f'n≥40%: {stat["high_n"]}; bias={stat["high_bias_pp"]:.1f} pp')
   ax.set_xlim(0,100);ax.set_ylim(0,100);ax.set_aspect('equal')
   ax.set_xlabel('UAV Musanga probability (%)')
  axes[0].set_ylabel('Satellite-predicted probability (%)')
  fig.suptitle(f'V4 strict forest-held-out satellite transfer | {scale} m | all predictor families')
  fig.tight_layout()
  fig.savefig(OUT/f'04_LOSO_scatter_all_{scale}m_{KEY}.png',dpi=180)
  plt.show()
 print('Output:',OUT)
